In [2]:
import os

from functools import partial
from typing import Callable

import evaluate

import numpy as np
import pandas as pd

from transformers import DataCollatorWithPadding, TrainingArguments, Trainer, pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from datasets import Dataset, ClassLabel
from evaluate import evaluator, combine

# Setup Data

In [3]:
dataset_path = "twitter_toxic_tweets.csv"
   
dataset = pd.read_csv(filepath_or_buffer=dataset_path)

In [4]:
dataset.head(n=2)

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...


In [5]:
data = dataset[["tweet", "label"]]
data.head(n=3)

,tweet,label
0,@user when a father is dysfunctional and is s...,0
1,@user @user thanks for #lyft credit i can't us...,0
2,bihday your majesty,0


In [6]:
data = data.rename({"tweet": "text"}, axis=1)

In [7]:
# Labels are already 0 (non-toxic) and 1 (toxic) — no replacement needed
data["label"].value_counts()

label
0    29720
1     2242
Name: count, dtype: int64

In [8]:
data

,text,label
0,@user when a father is dysfunctional and is s...,0
1,@user @user thanks for #lyft credit i can't us...,0
2,bihday your majesty,0
3,#model i love u take with u all the time in ...,0
4,factsguide: society now #motivation,0
...,...,...
31957,ate @user isz that youuu?ðððððð...,0
31958,to see nina turner on the airwaves trying to...,0
31959,listening to sad songs on a monday morning otw...,0
31960,"@user #sikh #temple vandalised in in #calgary,...",1


In [9]:
data["label"].value_counts()

label
0    29720
1     2242
Name: count, dtype: int64

# Setup Model

In [10]:
model_name = "distilbert-base-uncased"

# Dataset

## Load Dataset

In [24]:
data = data.dropna(subset=["text"]).reset_index(drop=True)
dataset = Dataset.from_pandas(df=data)
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 31962
})

## Labelling

In [23]:
classlabel = ClassLabel(num_classes=2, names=["non-toxic", "toxic"])
classlabel

ClassLabel(names=['non-toxic', 'toxic'])

In [25]:
dataset = dataset.cast_column(column="label", feature=classlabel)
dataset

Casting the dataset: 100%|██████████| 31962/31962 [00:00<00:00, 5334169.36 examples/s]


Dataset({
    features: ['text', 'label'],
    num_rows: 31962
})

In [14]:
dataset.features

{'text': Value('string'), 'label': ClassLabel(names=['non-toxic', 'toxic'])}

## Train Test Split

## Preprocess Data for Model

In [15]:
# Simple text preprocessing for the model
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    return text

In [26]:
sample = dataset[100]['text']
sample

'there are some truly sick ppl out there.   '

In [27]:
preprocess_text(sample)

'there are some truly sick ppl out there.'

In [28]:
dataset = dataset.map(function=lambda x: {"text": preprocess_text(x)}, input_columns="text")

Map: 100%|██████████| 31962/31962 [00:00<00:00, 67771.40 examples/s]


In [29]:
dataset[0]["text"]

'@user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction.   #run'

## Train Test Split

In [30]:
dataset = dataset.train_test_split(test_size=0.2, stratify_by_column="label")

In [31]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25569
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 6393
    })
})

## Tokenizer

In [32]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

In [33]:
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [36]:
dataset = dataset.map(
    function=lambda x: tokenizer(x, truncation=True, max_length=18),
    input_columns="text",
)

Map: 100%|██████████| 6393/6393 [00:00<00:00, 13268.31 examples/s]


In [37]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25569
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 6393
    })
})

## Model

In [38]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Use DataCollatorWithPadding to pad tokens and prepare batches, instead of doing it for the whole data, it is useful if you have a data doesn't fit into memory. 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, max_length=18)

# Metrics

In [40]:
f1 = evaluate.load("f1")

In [ ]:
def compute_metrics(eval_pred: np.ndarray, metric: evaluate.Metric):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="binary")

#returns a new callable where some parameters of func are fixed. so you
compute_metrics_fn = partial(compute_metrics, metric=f1)

# Training

## Training Args

In [44]:
training_args = TrainingArguments(
    output_dir=os.path.join(os.curdir, "data"),
    overwrite_output_dir=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_steps=50,
    logging_strategy="steps",
    logging_steps=50, 
    evaluation_strategy="steps",
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    save_steps=50,
    save_total_limit=1
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.21.0`: Please run `pip install transformers[torch]` or `pip install accelerate -U`

## Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_fn
)

: 

: 

## Train Model

In [ ]:
trainer.train()

: 

: 

## Evaluate Model

: 